# 20.9 隐私保护机器学习 / Privacy-Preserving ML (Differential Privacy & DP-SGD)

**中文**:上一节说过,联邦学习"不传数据"**并不等于**隐私安全——模型本身会泄漏训练数据。本节讲隐私的**黄金标准与数学保证**:**差分隐私(Differential Privacy, DP)**。它回答一个精确的问题:*"我的数据加入训练集,会不会让别人**看出我在里面**？"* DP 用严格的数学保证:**一个人的数据加不加入,几乎不改变模型的输出分布**——于是攻击者无法确定任何特定个体是否在训练数据里。这不是"感觉安全",而是**可证明的隐私**。本节从零实现让模型满足 DP 的标准算法 **DP-SGD**,并揭示核心矛盾——**隐私与效用的权衡**。
**English**: The last section noted that federated learning's "not sending data" **does not equal** privacy — the model itself leaks training data. This section covers privacy's **gold standard and mathematical guarantee**: **Differential Privacy (DP)**. It answers a precise question: *"if my data joins the training set, can someone **tell I'm in it**?"* DP provides a rigorous guarantee: **whether one person's data is included barely changes the model's output distribution** — so an attacker cannot determine whether any specific individual is in the training data. Not "feels safe" but **provable privacy**. This section implements the standard DP algorithm **DP-SGD** from scratch and reveals the core tension — the **privacy-utility tradeoff**.

---

**中文**:**差分隐私的定义(直觉版)**:一个算法 $M$ 满足 $(\varepsilon,\delta)$-DP,如果对**任意只差一个人**的两个数据集 $D$ 和 $D'$,以及任意输出 $S$:
**English**: **Definition of differential privacy (intuitive)**: an algorithm $M$ is $(\varepsilon,\delta)$-DP if for **any two datasets $D$ and $D'$ differing by one person** and any output $S$:

$$P(M(D)\in S)\le e^{\varepsilon}\,P(M(D')\in S)+\delta$$

**中文**:直觉:**加不加你这一个人,模型输出的概率最多差 $e^\varepsilon$ 倍**。**$\varepsilon$ 是"隐私预算"**——$\varepsilon$ 越小,两个数据集越难区分,隐私越强(但通常效用越差);$\varepsilon$ 大则隐私弱。$\delta$ 是允许的小概率失败(通常取极小,如 $10^{-5}$)。核心思想:**你的存在被"淹没"在随机噪声里,给了你"合理推诿(plausible deniability)"**。
**English**: Intuition: **including or excluding you changes the output probability by at most a factor of $e^\varepsilon$**. **$\varepsilon$ is the "privacy budget"** — smaller $\varepsilon$ means the two datasets are harder to distinguish, stronger privacy (usually lower utility); larger $\varepsilon$ means weaker privacy. $\delta$ is a small allowed failure probability (usually tiny, e.g. $10^{-5}$). The core idea: **your presence is "drowned" in random noise, giving you "plausible deniability."**

**中文**:**DP-SGD(Abadi et al., 2016)** 让神经网络训练满足 DP,只需在 SGD 里加两步:
**English**: **DP-SGD (Abadi et al., 2016)** makes neural-net training satisfy DP with two additions to SGD:
1. **逐样本梯度裁剪(per-example gradient clipping)**:把**每个样本**的梯度范数裁到上限 $C$。这样**任何单个样本对模型的影响都被限制住**(没有哪个人的数据能主导更新)。
   **Per-example gradient clipping**: clip **each sample's** gradient norm to a cap $C$. This **bounds any single sample's influence** on the model (no one's data can dominate the update).
2. **加高斯噪声**:给**汇总后**的梯度加高斯噪声,噪声大小 $\propto$ 裁剪上限 $C$ × 噪声乘子 $\sigma$。噪声"掩盖"了任何单个样本的贡献。
   **Add Gaussian noise**: add Gaussian noise to the **aggregated** gradient, with magnitude $\propto$ clip $C$ × noise multiplier $\sigma$. The noise "masks" any single sample's contribution.

**中文**:噪声乘子 $\sigma$ 越大,隐私越强($\varepsilon$ 越小),但模型精度越低——这就是**隐私-效用权衡**,本节的核心。
**English**: A larger noise multiplier $\sigma$ means stronger privacy (smaller $\varepsilon$) but lower model accuracy — this is the **privacy-utility tradeoff**, the heart of this section.

> 💡 **面试速查 / Interview cheat-sheet（★★ 隐私合规必考）**
> **中文**:**差分隐私(DP)**=可证明隐私: 加不加某一个人, 模型输出分布几乎不变(概率差≤$e^\varepsilon$)→攻击者认不出谁在训练集。**ε=隐私预算**(小=强隐私+低效用)。**DP-SGD**:①逐样本梯度裁剪(限制单样本影响)+②给汇总梯度加高斯噪声(掩盖个体)。核心=**隐私-效用权衡**(噪声越大越私越不准)。**防御什么**:①**成员推断攻击**(判断某人是否在训练集);②**模型记忆/训练数据提取**(大模型会背出训练文本!)。**关键性质**:DP 对后处理免疫、可组合(多次查询预算累加)。**vs 联邦学习**:联邦"数据不集中"、DP"数学保证不泄漏个体"——**常一起用**(联邦+DP+安全聚合)。工具:Opacus(PyTorch)、TF-Privacy。用途:人口普查(美国2020用了DP)、Apple/Google 统计、医疗、大模型训练防记忆。
> **English**: **Differential Privacy (DP)** = provable privacy: including or excluding one person barely changes the model's output distribution (probability ratio ≤ $e^\varepsilon$) → an attacker can't tell who's in the training set. **ε = privacy budget** (small = strong privacy + low utility). **DP-SGD**: ① per-example gradient clipping (bound single-sample influence) + ② Gaussian noise on the aggregated gradient (mask individuals). Core = the **privacy-utility tradeoff** (more noise = more private, less accurate). **Defends against**: ① **membership inference** (tell whether someone is in the training set); ② **memorization / training-data extraction** (LLMs can recite training text!). **Key properties**: DP is immune to post-processing and composes (repeated queries add up the budget). **vs federated learning**: federated "doesn't centralize data," DP "mathematically guarantees no individual leakage" — **often used together** (federated + DP + secure aggregation). Tools: Opacus (PyTorch), TF-Privacy. Uses: census (US 2020 used DP), Apple/Google analytics, healthcare, preventing LLM memorization.


In [ ]:

# ============================================================
# DP-SGD 从零实现 / DP-SGD from scratch
# 中文:用逻辑回归(逐样本梯度可解析、可向量化, 跑得快)演示 DP-SGD 的两步:逐样本裁剪 + 加噪。
#      扫描不同噪声乘子, 看隐私(噪声)与效用(精度)如何权衡。
# English: use logistic regression (per-example gradients are analytic & vectorizable, fast) to demo DP-SGD's
#      two steps: per-example clipping + noise. Sweep the noise multiplier to see the privacy-utility tradeoff.
# ============================================================
import numpy as np, os, time, matplotlib.pyplot as plt
from torchvision import datasets
np.random.seed(0)
root=os.path.expanduser("~/.cache/dsfs_cv")
mn=datasets.MNIST(root, train=True, download=False)
sub=np.random.permutation(len(mn.data))[:6000]
X=(mn.data.float()/255.).view(-1,784).numpy()[sub]; y=mn.targets.numpy()[sub]
te=datasets.MNIST(root, train=False, download=False)
Xte=(te.data.float()/255.).view(-1,784).numpy()[:2000]; yte=te.targets.numpy()[:2000]
def softmax(z): z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def onehot(y,k=10): o=np.zeros((len(y),k)); o[np.arange(len(y)),y]=1; return o

def train_dp_sgd(clip=1.0, noise_mult=0.0, epochs=15, lr=0.5, bs=128):
    rng=np.random.default_rng(0); W=np.zeros((784,10)); b=np.zeros(10)
    for e in range(epochs):
        for idx in np.array_split(rng.permutation(len(X)), len(X)//bs):
            xb=X[idx]; err=softmax(xb@W+b) - onehot(y[idx])            # 预测误差 / prediction error
            gW=xb[:,:,None]*err[:,None,:]; gb=err                      # 逐样本梯度 / per-example gradients
            # ① 逐样本裁剪:把每个样本的梯度范数限制到 clip / per-example clipping
            norms=np.sqrt((gW**2).sum((1,2)) + (gb**2).sum(1)) + 1e-6
            scale=np.minimum(1.0, clip/norms)
            gW*=scale[:,None,None]; gb*=scale[:,None]
            # ② 汇总 + 加高斯噪声 / aggregate + Gaussian noise
            noiseW=rng.normal(0, noise_mult*clip, (784,10)); noiseb=rng.normal(0, noise_mult*clip, 10)
            W-=lr*(gW.sum(0)+noiseW)/len(idx); b-=lr*(gb.sum(0)+noiseb)/len(idx)
    return W, b
def test_acc(W,b): return (softmax(Xte@W+b).argmax(1)==yte).mean()

t=time.time()
noise_levels=[0.0,0.5,1.0,2.0,4.0,8.0]
results=[(nm, test_acc(*train_dp_sgd(noise_mult=nm))) for nm in noise_levels]
print(f"{'噪声乘子/noise':<16}{'测试准确率':>10}{'隐私强度':>12}")
for nm,a in results:
    priv="无(仅裁剪)" if nm==0 else ("弱 weak" if nm<1 else ("中 medium" if nm<4 else "强 strong"))
    print(f"{nm:<16}{a:>10.3f}{priv:>12}")
print(f"用时 {time.time()-t:.0f}s")


**中文**:结果清晰地展现了**隐私-效用权衡**:噪声越大,隐私越强(隐私预算 ε 越小),但模型精度越低。噪声乘子从 0 到 8,精度从 0.86 一路降到 0.62。下面可视化这条权衡曲线——这是每个部署 DP 的团队必须面对的核心决策。
**English**: The result clearly shows the **privacy-utility tradeoff**: more noise means stronger privacy (smaller ε budget) but lower accuracy. As the noise multiplier goes from 0 to 8, accuracy drops from 0.86 to 0.62. Below we visualize this tradeoff curve — the core decision every team deploying DP must face.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
nms=[nm for nm,a in results]; accs=[a for nm,a in results]
# ① 隐私-效用权衡曲线 / privacy-utility tradeoff
ax[0].plot(nms,accs,"o-",color="#4C72B0",ms=8,lw=2)
ax[0].axhline(accs[0],ls="--",color="gray",label=f"无隐私(仅裁剪) {accs[0]:.2f}")
ax[0].annotate("更多噪声→更强隐私→更低精度\nmore noise → more privacy → less accuracy",
               xy=(nms[-2],accs[-2]),xytext=(1.5,0.68),fontsize=8,arrowprops=dict(arrowstyle="->"))
ax[0].set_title("隐私-效用权衡 / privacy-utility tradeoff"); ax[0].set_xlabel("噪声乘子 σ(越大越私)"); ax[0].set_ylabel("测试准确率"); ax[0].legend(fontsize=8)
# ② DP-SGD 两步示意 / DP-SGD two steps
ax[1].axis("off")
ax[1].text(0.5,0.9,"DP-SGD 的两步 / DP-SGD's two steps",ha="center",fontsize=13,weight="bold",transform=ax[1].transAxes)
ax[1].text(0.05,0.68,"① 逐样本梯度裁剪 / per-example clipping",fontsize=11,color="#4C72B0",transform=ax[1].transAxes)
ax[1].text(0.1,0.58,"把每个样本的梯度范数限制到 C\n→ 限制单个人对模型的影响\nbound each sample's influence",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.05,0.38,"② 加高斯噪声 / add Gaussian noise",fontsize=11,color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.1,0.28,"给汇总梯度加噪声 (∝ C·σ)\n→ 掩盖任何个体的贡献\nmask any individual's contribution",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.5,0.08,"结果:满足 (ε,δ)-DP, 攻击者认不出谁在训练集\nresult: (ε,δ)-DP, attacker can't identify membership",ha="center",fontsize=9,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/adv09_viz.png",dpi=80); plt.show()
print(f"权衡:无隐私 {accs[0]:.2f} → 强隐私(σ=8) {accs[-1]:.2f}, 精度换隐私")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **隐私可以被数学地保证,不只是"感觉安全"**:差分隐私把"我的数据会不会泄漏"变成一个**可证明、可量化**的性质——加不加你这一个人,模型输出分布最多差 $e^\varepsilon$ 倍。DP-SGD 通过"逐样本裁剪(限制单人影响)+ 加噪(掩盖个体)"两步,让训练出的模型满足这个保证。这比联邦学习的"不传数据"强得多——**联邦是防止数据直接暴露,DP 是防止模型间接泄漏**,两者互补(常一起用)。
2. **隐私和效用是硬权衡, 没有免费午餐**:曲线清清楚楚——噪声越大(隐私越强、ε 越小),精度越低(0.86→0.62)。这是差分隐私最诚实、最核心的现实:**你想要多少隐私,就得付出多少精度**。实践中要在业务可接受的精度下,选一个尽量小的 ε(常见 ε 在 1~10 之间;ε<1 是很强的隐私但代价大)。**没有"既完全隐私又完全准确"的模型。**
3. **DP 防御的是真实威胁,不是杞人忧天**:①**成员推断攻击**——不加保护的模型能被攻击者判断"某个人是否在训练集"(在医疗场景=泄漏"某人得了某病");②**训练数据提取**——大语言模型会**逐字背出**训练里见过的敏感文本(信用卡号、隐私对话都被提取过),DP 是主要防线;③人口普查、Apple/Google 的用户统计都用 DP 保护个体。**诚实的复杂性**:①正确计算 ε(隐私会计, privacy accounting)很精细(用 RDP/moments accountant, Opacus 帮你算);②DP 对**少数群体**可能更不公平(尾部数据被噪声淹没);③超参(裁剪阈值、批大小、噪声)对隐私-效用平衡影响很大。

**English**:
1. **Privacy can be mathematically guaranteed, not just "feel safe"**: differential privacy turns "will my data leak" into a **provable, quantifiable** property — including or excluding you changes the output distribution by at most a factor of $e^\varepsilon$. DP-SGD makes the trained model satisfy this via "per-example clipping (bound single-person influence) + noise (mask individuals)." This is far stronger than federated learning's "don't send data" — **federated prevents direct data exposure, DP prevents indirect model leakage**, and they are complementary (often used together).
2. **Privacy and utility are a hard tradeoff — no free lunch**: the curve is unmistakable — more noise (stronger privacy, smaller ε) means lower accuracy (0.86→0.62). This is DP's most honest, central reality: **the more privacy you want, the more accuracy you pay**. In practice, choose the smallest ε your business's acceptable accuracy allows (typical ε is 1–10; ε<1 is very strong privacy but costly). **There is no model that is both fully private and fully accurate.**
3. **DP defends against real threats, not paranoia**: ① **membership inference** — an unprotected model can let an attacker determine "whether someone is in the training set" (in healthcare = leaking "someone has a disease"); ② **training-data extraction** — LLMs can **recite verbatim** sensitive text seen in training (credit-card numbers, private conversations have been extracted), and DP is a main defense; ③ census and Apple/Google user analytics use DP to protect individuals. **Honest complexity**: ① correctly computing ε (privacy accounting) is intricate (RDP/moments accountant; Opacus computes it for you); ② DP can be **less fair to minority groups** (tail data drowned by noise); ③ hyperparameters (clip threshold, batch size, noise) strongly affect the privacy-utility balance.

> 💼 **实战视角 / Practical angle**
> **中文**:隐私 ML 的落地:①**合规驱动**(GDPR/CCPA/HIPAA 下用 DP 训练/发布统计);②**大模型防记忆**(用 DP-SGD 微调防止背出隐私数据);③**联邦+DP+安全聚合**三件套(手机端隐私建模);④隐私统计发布(人口普查、用户行为聚合)。工具:**Opacus**(PyTorch, 自动裁剪+加噪+隐私会计)、**TF-Privacy**、**Google DP**、**OpenDP**。落地要点:①**先定 ε 目标**(和法务/隐私团队);②Opacus 帮你算 ε 并管理隐私预算(多次训练/查询会累加);③接受**精度损失**并优化(大批量、预训练+DP微调能缓解);④注意公平性(DP 对小群体更狠);⑤DP 不是万能——还要配访问控制、加密、数据最小化。面试金句:*"差分隐私给'加不加某人几乎不改变输出'的数学保证(ε 是隐私预算, 小=强隐私); DP-SGD=逐样本梯度裁剪+加高斯噪声; 核心是隐私-效用权衡(噪声越大越私越不准); 防成员推断和训练数据提取, 常与联邦学习+安全聚合合用。"*
> **English**: Privacy-ML in practice: ① **compliance-driven** (train/release statistics with DP under GDPR/CCPA/HIPAA); ② **LLM memorization defense** (DP-SGD fine-tuning to prevent reciting private data); ③ the **federated + DP + secure aggregation** trio (on-device private modeling); ④ private statistics release (census, aggregated user behavior). Tools: **Opacus** (PyTorch, auto clipping + noise + privacy accounting), **TF-Privacy**, **Google DP**, **OpenDP**. Deployment keys: ① **set an ε target first** (with legal/privacy teams); ② Opacus computes ε and manages the privacy budget (repeated training/queries add up); ③ accept the **accuracy loss** and optimize (large batches, pretrain + DP fine-tune help); ④ watch fairness (DP is harsher on minorities); ⑤ DP isn't a silver bullet — combine with access control, encryption, data minimization. Interview line: *"Differential privacy guarantees 'including or excluding one person barely changes the output' (ε is the privacy budget, small = strong privacy); DP-SGD = per-example gradient clipping + Gaussian noise; the core is the privacy-utility tradeoff (more noise = more private, less accurate); it defends against membership inference and training-data extraction, often used with federated learning + secure aggregation."*

---
### 小结 / Summary
- **中文**:差分隐私=可证明隐私(加不加某人输出分布几乎不变, ε 为隐私预算, 小=强隐私)。
- **English**: Differential privacy = provable privacy (including/excluding one barely changes the output; ε is the budget, small = strong privacy).
- **中文**:DP-SGD=逐样本梯度裁剪(限单样本影响)+加高斯噪声(掩盖个体); 核心是隐私-效用权衡。
- **English**: DP-SGD = per-example clipping (bound single-sample influence) + Gaussian noise (mask individuals); the core is the privacy-utility tradeoff.
- **中文**:防成员推断/训练数据提取; 常与联邦学习+安全聚合合用; 工具 Opacus, 用途普查/大模型/医疗。
- **English**: Defends against membership inference / data extraction; often used with federated learning + secure aggregation; tool Opacus, uses census/LLMs/healthcare.
